# Circular Waves from a Point Source

This notebook builds the picture in three stages:

1. A single travelling sine wave along one ray.
2. The same travelling wave along separate rays.
3. A full circular ripple surface, with those directions highlighted.

The key idea is that a point source produces a disturbance whose phase depends on radial distance

$$r=\sqrt{x^2+y^2}$$

so the surface can be represented by

$$z(r,t)=A\sin(kr-\omega t).$$

The circular wavefronts correspond to equal phase — for example, the crests of the sine waves along every radial direction.


In [ ]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation

from IPython.display import HTML, display



# Optional widgets. These work in standard Jupyter and JupyterLite environments

# where ipywidgets is available.

try:

    import ipywidgets as widgets

    from ipywidgets import interact

    HAVE_WIDGETS = True

except ModuleNotFoundError:

    %pip install -q ipywidgets

    import ipywidgets as widgets

    from ipywidgets import interact

    HAVE_WIDGETS = True



plt.rcParams['figure.figsize'] = (8, 5)

plt.rcParams['animation.html'] = 'jshtml'


## 1. One travelling wave along a single ray

Imagine looking away from the source along the positive $x$-axis. The vertical displacement varies sinusoidally with distance from the source, and the pattern moves outward as time increases.


In [ ]:
def animate_single_ray(A=1.0, wavelength=2.0, speed=1.0, ray_length=10.0):

    k = 2*np.pi / wavelength

    omega = k * speed

    s = np.linspace(0, ray_length, 500)



    fig, ax = plt.subplots(figsize=(9, 4.5))



    line, = ax.plot([], [], lw=2)

    ax.axhline(0, lw=0.8)



    # Source

    ax.scatter([0], [0], s=70, zorder=5)



    # Red dots marking all crests

    crest_dots, = ax.plot([], [], 'ro', markersize=7, zorder=6)



    ax.set_xlim(0, ray_length)

    ax.set_ylim(-1.25*A, 1.25*A)

    ax.set_xlabel('distance from source, r')

    ax.set_ylabel('displacement')

    ax.set_title('Travelling sine wave along one ray')

    ax.grid(alpha=0.25)



    def update(frame):

        t = frame * 0.05



        z = A*np.sin(k*s - omega*t)

        line.set_data(s, z)



        # All crest positions satisfy:

        # k*r - omega*t = pi/2 + 2*pi*n

        n_min = -10

        n_max = 20



        crest_positions = []



        for n in range(n_min, n_max):

            r = (omega*t + np.pi/2 + 2*np.pi*n) / k



            if 0 <= r <= ray_length:

                crest_positions.append(r)



        crest_dots.set_data(

            crest_positions,

            [A] * len(crest_positions)

        )



        ax.set_title(

            f'Travelling sine wave along one ray   t = {t:.2f}'

        )



        return line, crest_dots



    anim = FuncAnimation(

        fig,

        update,

        frames=120,

        interval=50,

        blit=False

    )



    plt.close(fig)

    return HTML(anim.to_jshtml())





animate_single_ray()

## 2. Multiple radial sine waves

Now take exactly the same wave and place it along several directions from the same point source. Because the wave depends only on radial distance $r$, every direction has the same phase at the same distance from the source.


In [ ]:
def animate_rays(A=1.0, wavelength=2.0, speed=1.0, ray_length=7.0, n_rays=8):

    k = 2*np.pi / wavelength

    omega = k * speed

    s = np.linspace(0, ray_length, 350)

    angles = np.linspace(0, 2*np.pi, n_rays, endpoint=False)



    fig = plt.figure(figsize=(8, 7))

    ax = fig.add_subplot(111, projection='3d')



    lines = []



    for theta in angles:

        x = s*np.cos(theta)

        y = s*np.sin(theta)

        z = np.zeros_like(s)



        line, = ax.plot(

            x, y, z,

            linestyle='dashed',

            color='teal',

            lw=1

        )

        lines.append((line, theta))



    # Central bead

    bead = ax.scatter([0], [0], [0], s=60)



    # Red dots marking crests

    crest_dots = ax.scatter([], [], [], color='red', s=35)



    # Circular wavefront lines

    max_circles = int(ray_length / wavelength) + 2

    circle_lines = []



    for _ in range(max_circles):

        circle, = ax.plot([], [], [], color='red', lw=1)

        circle_lines.append(circle)



    ax.set_xlim(-ray_length, ray_length)

    ax.set_ylim(-ray_length, ray_length)

    ax.set_zlim(-5, 5)



    ax.set_xlabel('x')

    ax.set_ylabel('y')

    ax.set_zlabel('displacement')



    ax.set_title('Radial travelling waves')

    ax.view_init(elev=28, azim=-55)



    def update(frame):

        t = frame * 0.05



        # Wave displacement

        z = A*np.sin(k*s - omega*t)



        for line, theta in lines:

            x = s*np.cos(theta)

            y = s*np.sin(theta)



            line.set_data(x, y)

            line.set_3d_properties(z)



        # Move source bead

        z_bead = A*np.sin(-omega*t)

        bead._offsets3d = ([0], [0], [z_bead])



        # Find all visible crest radii

        crest_positions = []



        for n in range(-20, 30):

            r = (omega*t + np.pi/2 + 2*np.pi*n) / k



            if 0 <= r <= ray_length:

                crest_positions.append(r)



        # Red dots on each ray

        crest_x = []

        crest_y = []

        crest_z = []



        for theta in angles:

            for r in crest_positions:

                crest_x.append(r*np.cos(theta))

                crest_y.append(r*np.sin(theta))

                crest_z.append(A)



        crest_dots._offsets3d = (

            crest_x,

            crest_y,

            crest_z

        )



        # Horizontal circles joining corresponding crest dots

        phi = np.linspace(0, 2*np.pi, 200)



        for i, circle in enumerate(circle_lines):



            if i < len(crest_positions):

                r = crest_positions[i]



                x_circle = r*np.cos(phi)

                y_circle = r*np.sin(phi)

                z_circle = np.full_like(phi, A)



                circle.set_data(x_circle, y_circle)

                circle.set_3d_properties(z_circle)



            else:

                circle.set_data([], [])

                circle.set_3d_properties([])



        ax.set_title(

            f'Radial travelling waves   t = {t:.2f}'

        )



        return (

            [item[0] for item in lines]

            + [bead, crest_dots]

            + circle_lines

        )



    anim = FuncAnimation(

        fig,

        update,

        frames=120,

        interval=50,

        blit=False

    )



    plt.close(fig)



    return HTML(anim.to_jshtml())





animate_rays()

## 4. Top view: wavefronts and crests

This view makes the connection especially clear. Bright and dark rings are equal-phase circles. The highlighted rays simply cut through those same circular wavefronts.


In [ ]:
def animate_top_view(A=1.0, wavelength=2.0, speed=1.0, extent=7.0, n_rays=8):

    k = 2*np.pi / wavelength

    omega = k * speed



    x = np.linspace(-extent, extent, 250)

    y = np.linspace(-extent, extent, 250)

    X, Y = np.meshgrid(x, y)

    R = np.sqrt(X**2 + Y**2)

    angles = np.linspace(0, 2*np.pi, n_rays, endpoint=False)



    fig, ax = plt.subplots(figsize=(7, 7))



    def update(frame):

        ax.clear()

        t = frame * 0.06

        Z = A*np.sin(k*R - omega*t)

        ax.contourf(X, Y, Z, levels=30, cmap='viridis')

        ax.contour(X, Y, Z, levels=[0.95*A], colors='white', linewidths=1.5)



        for theta in angles:

            ax.plot([0, extent*np.cos(theta)], [0, extent*np.sin(theta)], lw=1.4)



        ax.scatter([0], [0], s=60)

        ax.set_aspect('equal')

        ax.set_xlim(-extent, extent)

        ax.set_ylim(-extent, extent)

        ax.set_xlabel('x')

        ax.set_ylabel('y')

        ax.set_title(f'Top view: circular wavefronts   t = {t:.2f}')



    anim = FuncAnimation(fig, update, frames=100, interval=70, blit=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())



animate_top_view()
